In [6]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import os
import re
import librosa
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


AudioSegment.converter = "D:/Github/phone-cleaner/bin/ffmpeg.exe"
AudioSegment.ffprobe = "D:/Github/phone-cleaner/bin/ffprobe.exe"
input_folder = "./phoneme-Samples/Glossika/wav-no-music/"
output_folder = "./phoneme-Samples/Glossika/tight-snips/"

# Create output directory if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

volume_threshold = -25  # dB
lead_time = 200  # milliseconds
follow_time = 200  # milliseconds

def find_name(input_string):
    # Use a regular expression to find the content within the first pair of square brackets
    match = re.search(r'\] ([^\]]+)\.', input_string)
    
    # If a match is found, trim leading and trailing spaces
    if match:
        return match.group(1).strip()
    else:
        return None
    
def find_bracket_contents(input_string):
    # Use a regular expression to find the content within the first pair of square brackets
    match = re.search(r'\[([^\]]+)\]', input_string)
    
    # If a match is found, trim leading and trailing spaces
    if match:
        return match.group(1).strip()
    else:
        return None

def find_segments_by_cluster(cluster_labels, times):
    """
    Find continuous segments for each cluster
    """
    segments_by_cluster = {}
    
    # Find continuous segments of each cluster
    current_cluster = None
    segment_start = None
    
    for i, cluster_id in enumerate(cluster_labels):
        if cluster_id != current_cluster:
            # End previous segment if it exists
            if current_cluster is not None and segment_start is not None:
                if current_cluster not in segments_by_cluster:
                    segments_by_cluster[current_cluster] = []
                segments_by_cluster[current_cluster].append((times[segment_start], times[i-1]))
            
            # Start new segment
            current_cluster = cluster_id
            segment_start = i
    
    # Handle final segment
    if current_cluster is not None and segment_start is not None:
        if current_cluster not in segments_by_cluster:
            segments_by_cluster[current_cluster] = []
        segments_by_cluster[current_cluster].append((times[segment_start], times[-1]))
    
    # Merge adjacent segments of the same cluster that are very close
    for cluster_id in segments_by_cluster:
        if len(segments_by_cluster[cluster_id]) > 1:
            merged = []
            current_seg = segments_by_cluster[cluster_id][0]
            
            for next_seg in segments_by_cluster[cluster_id][1:]:
                # If gap between segments is less than 50ms, merge them
                if next_seg[0] - current_seg[1] < 0.05:
                    current_seg = (current_seg[0], next_seg[1])
                else:
                    merged.append(current_seg)
                    current_seg = next_seg
            
            merged.append(current_seg)
            segments_by_cluster[cluster_id] = merged
    
    return segments_by_cluster

def plot_kmeans_results(y, times, rms, zcr, spectral_centroid, cluster_labels, 
                       cluster_classifications, segment_type, audio_filename, duration, segment_idx):
    """
    Plot the K-means clustering results for a segment with cluster highlighting
    """
    plt.figure(figsize=(15, 10))
    
    # Find segments by cluster for coloring
    segments_by_cluster = find_segments_by_cluster(cluster_labels, times)
    
    # Waveform with cluster coloring
    plt.subplot(4, 1, 1)
    plt.plot(np.linspace(0, duration, len(y)), y, alpha=0.7, color='gray')
    plt.title(f'Segment {segment_idx} - Detected as: {segment_type.upper()}')
    plt.ylabel('Amplitude')
    
    # Add colored regions for each cluster
    unique_clusters = np.unique(cluster_labels)
    cluster_colors = plt.cm.tab10(np.linspace(0, 1, len(unique_clusters)))
    color_map = {cluster_id: cluster_colors[i] for i, cluster_id in enumerate(unique_clusters)}
    
    for cluster_id, segments in segments_by_cluster.items():
        color = color_map[cluster_id]
        label_name = cluster_classifications[cluster_id]['label']
        
        for i, (start, end) in enumerate(segments):
            # Only add label for first segment of each cluster to avoid duplicate legends
            label = f'Cluster {cluster_id}: {label_name}' if i == 0 else ""
            plt.axvspan(start, end, alpha=0.3, color=color, label=label)
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # RMS Energy with clusters
    plt.subplot(4, 1, 2)
    
    for i, cluster_id in enumerate(unique_clusters):
        mask = cluster_labels == cluster_id
        label_name = cluster_classifications[cluster_id]['label']
        plt.scatter(times[mask], rms[mask], c=[cluster_colors[i]], 
                   label=f'Cluster {cluster_id}: {label_name}', alpha=0.7, s=15)
    
    plt.plot(times, rms, color='black', alpha=0.3, linewidth=0.5)
    plt.title('RMS Energy by Cluster')
    plt.ylabel('Energy')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Zero Crossing Rate
    plt.subplot(4, 1, 3)
    for i, cluster_id in enumerate(unique_clusters):
        mask = cluster_labels == cluster_id
        plt.scatter(times[mask], zcr[mask], c=[cluster_colors[i]], alpha=0.7, s=15)
    
    plt.plot(times, zcr, color='black', alpha=0.3, linewidth=0.5)
    plt.title('Zero Crossing Rate by Cluster')
    plt.ylabel('ZCR')
    
    # Spectral Centroid
    plt.subplot(4, 1, 4)
    for i, cluster_id in enumerate(unique_clusters):
        mask = cluster_labels == cluster_id
        plt.scatter(times[mask], spectral_centroid[mask], c=[cluster_colors[i]], alpha=0.7, s=15)
    
    plt.plot(times, spectral_centroid, color='black', alpha=0.3, linewidth=0.5)
    plt.title('Spectral Centroid by Cluster')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (s)')
    
    plt.tight_layout()
    
    # Save plot
    base_filename = os.path.splitext(audio_filename)[0]
    plot_path = os.path.join(output_folder, f"{base_filename}_segment_{segment_idx}_{segment_type}_kmeans.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"K-means plot saved: {plot_path}")

def analyze_segment_with_kmeans(audio_data, sr, audio_filename="", segment_idx=0, save_plot=True):
    """
    Analyze audio segment using kmeans to determine if it's isolated C, CV, VCV, or VC
    """
    # Calculate frame parameters
    frame_length = int(sr * 0.025)  # 25ms frames
    hop_length = int(sr * 0.010)    # 10ms hop
    
    # Extract features
    rms = librosa.feature.rms(y=audio_data, frame_length=frame_length, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(y=audio_data, frame_length=frame_length, hop_length=hop_length)[0]
    spectral_centroid = librosa.feature.spectral_centroid(y=audio_data, sr=sr, hop_length=hop_length)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio_data, sr=sr, hop_length=hop_length)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=audio_data, sr=sr, hop_length=hop_length)[0]
    spectral_flatness = librosa.feature.spectral_flatness(y=audio_data, hop_length=hop_length)[0]
    mfcc = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=13, hop_length=hop_length)
    
    # Time axis
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)
    duration = len(audio_data) / sr
    
    # Combine features into feature matrix
    features = np.vstack([
        rms,
        zcr,
        spectral_centroid,
        spectral_bandwidth,
        spectral_rolloff,
        spectral_flatness,
        mfcc[:5]  # Use first 5 MFCCs
    ]).T
    
    # Handle any NaN values
    features = np.nan_to_num(features, nan=0.0)

    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Use 2 or 3 clusters based on segment length
    n_clusters = 2 #min(3, len(features))
    if n_clusters < 2:
        return "iso"
        
    # Apply K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features_scaled)
    
    # Analyze clusters
    cluster_info = {}
    for i in range(n_clusters):
        cluster_mask = cluster_labels == i
        cluster_indices = np.where(cluster_mask)[0]
        
        if len(cluster_indices) > 0:
            avg_rms = np.mean(rms[cluster_indices])
            avg_zcr = np.mean(zcr[cluster_indices])
            avg_centroid = np.mean(spectral_centroid[cluster_indices])
            avg_time_position = np.mean(times[cluster_indices])
            
            cluster_info[i] = {
                'avg_rms': avg_rms,
                'avg_zcr': avg_zcr,
                'avg_centroid': avg_centroid,
                'avg_time_position': avg_time_position,
                'frame_count': len(cluster_indices)
            }
    
    # Create cluster classifications for plotting
    cluster_classifications = {}
    for cluster_id, info in cluster_info.items():
        rms_val = info['avg_rms']
        zcr_val = info['avg_zcr']
        centroid_val = info['avg_centroid']
        
        cluster_classifications[cluster_id] = {
            'label': f'RMS:{rms_val:.3f} ZCR:{zcr_val:.3f}',
            'group': f'cluster_{cluster_id}'
        }
    
    # Classify segment type based on clusters
    segment_type = "unk"  # Default
    
    if n_clusters == 2:
        # Find highest RMS (vowel) and earliest timing (consonant)
        max_rms = 0
        vowel_cluster = None
        min_time = float('inf')
        consonant_cluster = None
        
        for cluster_id, info in cluster_info.items():
            if info['avg_rms'] > max_rms:
                max_rms = info['avg_rms']
                vowel_cluster = cluster_id
            if info['avg_time_position'] < min_time:
                min_time = info['avg_time_position']
                consonant_cluster = cluster_id
        
        # Determine if it's CV or VC based on cluster sequence
        if vowel_cluster != consonant_cluster:
            # Check temporal order
            vowel_time = cluster_info[vowel_cluster]['avg_time_position']
            consonant_time = cluster_info[consonant_cluster]['avg_time_position']
            
            if consonant_time < vowel_time:
                segment_type = "pre"  # CV pattern
            else:
                segment_type = "post"  # VC pattern
        else:
            segment_type = "unk"  # Single dominant cluster type
            
    elif n_clusters == 3:
        # Three clusters - identify consonant and vowel clusters
        # Find the cluster with highest RMS as vowel
        max_rms = 0
        vowel_cluster = None
        for cluster_id, info in cluster_info.items():
            if info['avg_rms'] > max_rms:
                max_rms = info['avg_rms']
                vowel_cluster = cluster_id
        
        # Get temporal positions of all clusters
        cluster_times = [(cluster_id, info['avg_time_position']) for cluster_id, info in cluster_info.items()]
        cluster_times.sort(key=lambda x: x[1])  # Sort by time
        
        # Check if vowel is in the middle position (VCV pattern)
        middle_cluster = cluster_times[1][0]  # Cluster in middle position temporally
        
        if middle_cluster == vowel_cluster:
            # Vowel is in middle - this is VCV (vowel-consonant-vowel) pattern
            segment_type = "med"  # VCV pattern
        else:
            # Vowel is not in middle, check if it's at start or end
            first_cluster = cluster_times[0][0]
            if first_cluster == vowel_cluster:
                segment_type = "post"  # VCC or VC pattern - vowel at start
            else:
                segment_type = "pre"  # CCV or CV pattern - vowel at end
    
    # Generate plot if requested
    if save_plot and len(audio_filename) > 0:
        plot_kmeans_results(audio_data, times, rms, zcr, spectral_centroid, cluster_labels,
                           cluster_classifications, segment_type, audio_filename, duration, segment_idx)
    
    return segment_type

In [ ]:
#Clipping Glossika with K-means Analysis
for filename in os.listdir(input_folder):
    
    if filename.endswith(".wav"):
        audio_path = os.path.join(input_folder, filename)
        # Use Unicode string
        audio_path = audio_path
        
        
        print(audio_path)
         # Print the audio path to debug
        print("Processing file:", audio_path)
        
        # Convert the file to a standard format
        temp_audio_path = os.path.join(output_folder, "temp_output.wav")
        #start_time = "00:00:25"  # 25 seconds
        #end_time = "00:00:50"    # 50 seconds
        
        #conversion_command = f'ffmpeg.exe -y -ss {start_time} -to {end_time} -i "{audio_path}" -acodec pcm_s16le -ar 44100 "{temp_audio_path}"'
        conversion_command = f'ffmpeg.exe -y "{audio_path}" -acodec pcm_s16le -ar 44100 "{temp_audio_path}"'


        print(f"Running command: {conversion_command}")
        result = os.system(conversion_command)
        print(f"Command result: {result}")
        
        # Check if temp file was created
        if not os.path.exists(temp_audio_path):
            print(f"Error: temp file not created at {temp_audio_path}")
            continue
        
        try:
            # Load the converted audio file
            audio = AudioSegment.from_file(temp_audio_path, format="wav")
        except Exception as e:
            print("Error loading audio file:", e)
            continue
        
        # Detect nonsilent segments
        nonsilent_ranges = detect_nonsilent(audio, min_silence_len=300, silence_thresh=volume_threshold)
        phoneme = find_bracket_contents(filename)
        name = find_name(filename)
        
        for i, (start, end) in enumerate(nonsilent_ranges):
            start = max(0, start - lead_time)
            end = min(len(audio), end + follow_time)
            clip = audio[start:end]
            
            # Convert AudioSegment to numpy array for analysis
            audio_samples = np.array(clip.get_array_of_samples())
            if clip.channels == 2:
                audio_samples = audio_samples.reshape((-1, 2))
                audio_samples = audio_samples.mean(axis=1)  # Convert to mono
            
            # Convert to float and normalize
            audio_samples = audio_samples.astype(np.float32)
            if audio_samples.max() > 1.0:
                audio_samples = audio_samples / (2**15)  # Normalize 16-bit audio
            
            # Analyze with k-means to determine segment type (with plotting)
            try:
                segment_type = analyze_segment_with_kmeans(
                    audio_samples, 
                    clip.frame_rate, 
                    audio_filename=filename, 
                    segment_idx=i, 
                    save_plot=True
                )
                print(f"Segment {i}: Detected as {segment_type}")
            except Exception as e:
                print(f"Error analyzing segment {i}: {e}")
                segment_type = "unk"  # Default fallback
            
            # Generate filename based on acoustic analysis
            output_filename = f"{os.path.splitext(filename)[0]}_clip_{i}.wav"
            
            if segment_type == "iso":
                output_filename = f"iso_[{phoneme}]_{phoneme}_{name}.wav"
            elif segment_type == "pre":
                output_filename = f"pre_[{phoneme}]_{phoneme}ə_{name}.wav"
            elif segment_type == "med":
                output_filename = f"med_[{phoneme}]_ə{phoneme}ə_{name}.wav"
            elif segment_type == "post":
                output_filename = f"post_[{phoneme}]_ə{phoneme}_{name}.wav"
                
            output_path = os.path.join(output_folder, output_filename)
            
            # Only export the first 4 segments (as before)
            if 0 <= i <= 3:
                # Print the output path to debug
                print("Exporting clip to:", output_path)

                try:
                    # Export clip with Unicode handling
                    if ("_clean" in output_path):
                        output_path = output_path.replace("_clean", "")
                    clip.export(output_path, format="wav")
                    print(f"Successfully exported: {output_filename} (Type: {segment_type})")
                except Exception as e:
                    print("Error exporting file:", e)
                    continue

                
# Segment detection legend:
# iso: Isolated consonant/phoneme
# pre: Pre-vowel (CV pattern)  
# med: Medial (VCV pattern) - vowel-consonant-vowel
# post: Post-vowel (VC pattern)

# Each processed segment will generate:
# 1. Audio file with classified name (iso_, pre_, med_, or post_)
# 2. PNG plot showing kmeans analysis with waveform, RMS, ZCR, and spectral centroid

./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav
Processing file: ./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav
Running command: ffmpeg.exe -y "./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav" -acodec pcm_s16le -ar 44100 "./phoneme-Samples/Glossika/tight-snips/temp_output.wav"
Command result: 1
Error: temp file not created at ./phoneme-Samples/Glossika/tight-snips/temp_output.wav
./phoneme-Samples/Glossika/wav-no-music/[ bʱ ] voiced aspirated bilabial stop_clean.wav
Processing file: ./phoneme-Samples/Glossika/wav-no-music/[ bʱ ] voiced aspirated bilabial stop_clean.wav
Running command: ffmpeg.exe -y "./phoneme-Samples/Glossika/wav-no-music/[ bʱ ] voiced aspirated bilabial stop_clean.wav" -acodec pcm_s16le -ar 44100 "./phoneme-Samples/Glossika/tight-snips/temp_output.wav"
Command result: 1
Error: temp file not created at ./phoneme-Samples/Glossika/tight-snips/temp_

KeyboardInterrupt: 

In [4]:
#Clipping IPA

# Waiting to clean up inconsistent segments
verbose = False

input_folder = "C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV/"
output_folder = "C:/Github/phone-cleaner/phoneme-Samples/Glossika/tight-snips/"

for filename in os.listdir(input_folder):
    
    if filename.endswith(".wav"):
        audio_path = os.path.join(input_folder, filename)
        # Use Unicode string
        audio_path = audio_path
        
        
        print(audio_path)
         # Print the audio path to debug
        if verbose: 
            print("Processing file:", audio_path)
        
        # Convert the file to a standard format
        temp_audio_path = os.path.join(output_folder, "temp_output.wav")
        
        conversion_command = f'ffmpeg -y -i "{audio_path}" -acodec pcm_s16le -ar 44100 "{temp_audio_path}"'
        
        os.system(conversion_command)
        
        try:
            # Load the converted audio file
            audio = AudioSegment.from_file(temp_audio_path, format="wav")
        except Exception as e:
            print("Error loading audio file:", e)
            continue
        
        # Detect nonsilent segments
        nonsilent_ranges = detect_nonsilent(audio, min_silence_len=100, silence_thresh=volume_threshold)
        phoneme = find_bracket_contents(filename)
        name = find_name(filename)
        for i, (start, end) in enumerate(nonsilent_ranges):
            start = max(0, start - lead_time)
            end = min(len(audio), end + follow_time)
            clip = audio[start:end]
            output_filename = f"{os.path.splitext(filename)[0]}_clip_{i}.wav"
            if i == 0:
                output_filename = f"iso_{filename}_clip_{i}" ############Filename
                if verbose:
                    print(output_filename)
            if i == 1:
                output_filename = f"x_{filename}_clip_{i}"
                print(output_filename)
            if i == 2:
                output_filename = f"x_{filename}_clip_{i}"
                print(output_filename)
            output_path = os.path.join(output_folder, output_filename)
            if ( 0 <= i <= 3):
                # Print the output path to debug
                if verbose:
                    print("Exporting clip to:", output_path)
                try:
                    # Export clip with Unicode handling
                    if True == True:
                        clip.export(output_path, format="wav")
                except Exception as e:
                    print("Error exporting file:", e)
                    continue


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV/'

In [ ]:
import os
import subprocess
import json

def get_audio_details(file_path):
    command = [
        'ffprobe',
        '-v', 'error',
        '-show_format',
        '-show_streams',
        '-print_format', 'json',
        file_path
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error getting details for {file_path}: {result.stderr}")
        return None
    
    return json.loads(result.stdout)

def convert_webm_to_wav(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Get a list of all .webm files in the input folder
    webm_files = [f for f in os.listdir(input_folder) if f.endswith('.webm')]

    # Iterate over each .webm file and convert to WAV
    for webm_file in webm_files:
        input_path = os.path.join(input_folder, webm_file)
        output_file = os.path.splitext(webm_file)[0] + '.wav'
        output_path = os.path.join(output_folder, output_file)

        # Ensure paths are correctly formatted for subprocess
        input_path = os.path.abspath(input_path)
        output_path = os.path.abspath(output_path)

        # FFMPEG command for conversion
        command = [
            'ffmpeg',
            '-i', input_path,
            '-acodec', 'pcm_s16le',  # 16-bit little-endian PCM audio
            '-ar', '44100',
            '-f', 'wav',
            output_path
        ]

        # Print command for debugging
        print(f"Running command: {' '.join(command)}")

        # Execute the FFMPEG command
        result = subprocess.run(command, capture_output=True, text=True)

        # Print stdout and stderr for debugging
        print("stdout:", result.stdout)
        print("stderr:", result.stderr)

        # Check if the output file was created successfully
        if os.path.exists(output_path):
            print(f'Conversion complete: {webm_file} -> {output_file}')
            
            # Get and print audio details using ffprobe
            details = get_audio_details(output_path)
            if details:
                print(f"Audio details for {output_file}:")
                print(json.dumps(det


In [1]:
## Convert IPA to Wav

import os
from pydub import AudioSegment

def convert_mp3_to_wav(input_folder, output_folder):
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Loop through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith(".mp3"):
            mp3_path = os.path.join(input_folder, filename)
            wav_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".wav")
            
            # Load the MP3 file
            audio = AudioSegment.from_mp3(mp3_path)
            
            # Export as WAV with PCM 16-bit LE encoding
            audio.export(wav_path, format="wav", codec="pcm_s16le")

            print(f"Converted {filename} to {wav_path}")

paths = [
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JE", 
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JW",
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JH",
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/PL"
]

output_folder = "C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV"
for input_folder in paths:
    convert_mp3_to_wav(input_folder, output_folder)






Converted iso_[a]_a_OPEN FRONT UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a]_a_OPEN FRONT UNROUNDED VOWEL_IPAJE.wav
Converted iso_[a˞]_a˞_RHOTICITY_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a˞]_a˞_RHOTICITY_2_IPAJE.wav
Converted iso_[a̤]_a̤_BREATHY VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a̤]_a̤_BREATHY VOICED_2_IPAJE.wav
Converted iso_[a̰]_a̰_CREAKY VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a̰]_a̰_CREAKY VOICED_2_IPAJE.wav
Converted iso_[b]_b_VOICED BILABIAL PLOSIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b]_b_VOICED BILABIAL PLOSIVE_IPAJE.wav
Converted iso_[b̤]_b̤_BREATHY VOICED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̤]_b̤_BREATHY VOICED_1_IPAJE.wav
Converted iso_[b̰]_b̰_CREAKY VOICED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̰]_b̰_CREAKY VOICED_1_IPAJ

Converted iso_[tʷ]_tʷ_LABIALIZED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tʷ]_tʷ_LABIALIZED_1_IPAJE.wav
Converted iso_[tʼ]_tʼ_DENTAL or ALVEOLAR EJECTIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tʼ]_tʼ_DENTAL or ALVEOLAR EJECTIVE_IPAJE.wav
Converted iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJE.wav
Converted iso_[t̪]_t̪_DENTAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̪]_t̪_DENTAL_1_IPAJE.wav
Converted iso_[t̬]_t̬_VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̬]_t̬_VOICED_2_IPAJE.wav
Converted iso_[t̺]_t̺_APICAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̺]_t̺_APICAL_1_IPAJE.wav
Converted iso_[t̻]_t̻_LAMINAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̻]_t̻_LAMINAL_1_IPAJE.wav
Converted iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJE.mp3 to C:/G

Converted iso_[ɢ]_ɢ_VOICED UVULAR PLOSIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɢ]_ɢ_VOICED UVULAR PLOSIVE_IPAJE.wav
Converted iso_[ɤ]_ɤ_CLOSE-MID BACK UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɤ]_ɤ_CLOSE-MID BACK UNROUNDED VOWEL_IPAJE.wav
Converted iso_[ɥ]_ɥ_VOICED LABIAL-PALATAL APPROXIMANT_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɥ]_ɥ_VOICED LABIAL-PALATAL APPROXIMANT_IPAJE.wav
Converted iso_[ɦ]_ɦ_VOICED GLOTTAL FRICATIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɦ]_ɦ_VOICED GLOTTAL FRICATIVE_IPAJE.wav
Converted iso_[ɧ]_ɧ_VOICELESS POSTALVEOLAR-VELAR FRICATIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɧ]_ɧ_VOICELESS POSTALVEOLAR-VELAR FRICATIVE_IPAJE.wav
Converted iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAJE.wa

Converted iso_[b̤a̤]_b̤a̤_BREATHY VOICED_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̤a̤]_b̤a̤_BREATHY VOICED_IPAJW.wav
Converted iso_[b̰a̰]_b̰a̰_CREAKY VOICED_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̰a̰]_b̰a̰_CREAKY VOICED_IPAJW.wav
Converted iso_[c]_c_VOICELESS PALATAL PLOSIVE_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[c]_c_VOICELESS PALATAL PLOSIVE_IPAJW.wav
Converted iso_[d]_d_VOICED DENTAL or ALVEOLAR PLOSIVE_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d]_d_VOICED DENTAL or ALVEOLAR PLOSIVE_IPAJW.wav
Converted iso_[dʰ]_dʰ_ASPIRATED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʰ]_dʰ_ASPIRATED_2_IPAJW.wav
Converted iso_[dʲ]_dʲ_PALATALIZED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʲ]_dʲ_PALATALIZED_2_IPAJW.wav
Converted iso_[dʷ]_dʷ_LABIALIZED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʷ]_dʷ_LAB

Converted iso_[tˠ]_tˠ_VELARIZED_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˠ]_tˠ_VELARIZED_1_IPAJW.wav
Converted iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJW.wav
Converted iso_[t̪]_t̪_DENTAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̪]_t̪_DENTAL_1_IPAJW.wav
Converted iso_[t̺]_t̺_APICAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̺]_t̺_APICAL_1_IPAJW.wav
Converted iso_[t̻]_t̻_LAMINAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̻]_t̻_LAMINAL_1_IPAJW.wav
Converted iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJW.wav
Converted iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAJW.wav
Converted iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAJW.mp3 to C:/Git

...

Converted iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAJH.wav
Converted iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAJH.wav
Converted iso_[z]_z_VOICED ALVEOLAR FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[z]_z_VOICED ALVEOLAR FRICATIVE_IPAJH.wav
Converted iso_[æ]_æ_NEAR-OPEN FRONT UNROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[æ]_æ_NEAR-OPEN FRONT UNROUNDED VOWEL_IPAJH.wav
Converted iso_[ç]_ç_VOICELESS PALATAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ç]_ç_VOICELESS PALATAL FRICATIVE_IPAJH.wav
Converted iso_[ð]_ð_VOICED DENTAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ð]_ð_VOICED DENTAL FRICATIVE_IPAJH.wav
Converted iso_[ø]_ø_CLOSE-MID FRONT ROUNDED VOWEL_IP

Converted iso_[ɴ]_ɴ_VOICED UVULAR NASAL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɴ]_ɴ_VOICED UVULAR NASAL_IPAJH.wav
Converted iso_[ɶ]_ɶ_OPEN FRONT ROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɶ]_ɶ_OPEN FRONT ROUNDED VOWEL_IPAJH.wav
Converted iso_[ɸ]_ɸ_VOICELESS BILABIAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɸ]_ɸ_VOICELESS BILABIAL FRICATIVE_IPAJH.wav
Converted iso_[ɹ]_ɹ_VOICED DENTAL or ALVEOLAR APPROXIMANT_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɹ]_ɹ_VOICED DENTAL or ALVEOLAR APPROXIMANT_IPAJH.wav
Converted iso_[ɹ̝]_ɹ̝_RAISED_2_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɹ̝]_ɹ̝_RAISED_2_IPAJH.wav
Converted iso_[ɺ]_ɺ_VOICED ALVEOLAR LATERAL FLAP_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɺ]_ɺ_VOICED ALVEOLAR LATERAL FLAP_IPAJH.wav
Converted iso_[ɻ]_ɻ_VOICED RETROFLEX APPROXIMANT_IPAJH.mp3 to C:/Github/

Converted iso_[d̥]_d̥_VOICELESS_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̥]_d̥_VOICELESS_2_IPAPL.wav
Converted iso_[d̪]_d̪_DENTAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̪]_d̪_DENTAL_2_IPAPL.wav
Converted iso_[d̺]_d̺_APICAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̺]_d̺_APICAL_2_IPAPL.wav
Converted iso_[d̻]_d̻_LAMINAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̻]_d̻_LAMINAL_2_IPAPL.wav
Converted iso_[d̼]_d̼_LINGUOLABIAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̼]_d̼_LINGUOLABIAL_2_IPAPL.wav
Converted iso_[e]_e_CLOSE-MID FRONT UNROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[e]_e_CLOSE-MID FRONT UNROUNDED VOWEL_IPAPL.wav
Converted iso_[eː]_eː_LONG_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[eː]_eː_LONG_IPAPL.wav
Converted iso_[eˑ]_eˑ_HALF-LONG_IPAPL.mp3 to C:/Github/phone-cle

Converted iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAPL.wav
Converted iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAPL.wav
Converted iso_[u̟]_u̟_ADVANCED_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[u̟]_u̟_ADVANCED_IPAPL.wav
Converted iso_[v]_v_VOICED LABIODENTAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[v]_v_VOICED LABIODENTAL FRICATIVE_IPAPL.wav
Converted iso_[w]_w_VOICED LABIAL-VELAR APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[w]_w_VOICED LABIAL-VELAR APPROXIMANT_IPAPL.wav
Converted iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAPL.wav
Converted iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme

Converted iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAPL.wav
Converted iso_[ɬ]_ɬ_VOICELESS DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɬ]_ɬ_VOICELESS DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.wav
Converted iso_[ɭ]_ɭ_VOICED RETROFLEX LATERAL APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɭ]_ɭ_VOICED RETROFLEX LATERAL APPROXIMANT_IPAPL.wav
Converted iso_[ɮ]_ɮ_VOICED DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɮ]_ɮ_VOICED DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.wav
Converted iso_[ɰ]_ɰ_VOICED VELAR APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɰ]_ɰ_VOICED VELAR APPROXIMANT_IPAPL.wav
Converted iso_[ɳ]_ɳ_VOICED RETROFLEX NASAL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IP